Daily Challenge : How to Finetune LLMs with LoRA
https://octopus.developers.institute/courses/collection/117/course/659/section/1788/chapter/4500

Parameter-Efficient Fine-Tuning (PEFT) methods, like LoRA, address the challenges of fine-tuning large language models (LLMs) by only updating a small subset of the model’s parameters. This approach significantly reduces computational and storage costs, making LLM fine-tuning more accessible. PEFT techniques allow developers to adapt pre-trained models to specific tasks without retraining the entire model, leading to faster development cycles and reduced resource consumption.
You will implement it for this challenge.

👩‍🏫 👩🏿‍🏫 What You’ll learn
        # How to apply Low-Rank Adaptation (LoRA) to a pre-trained language model.
        # How to fine-tune a LoRA-adapted model using the Hugging Face PEFT library.
        # How to save and load a fine-tuned LoRA model.
        # How to perform inference using a fine-tuned LoRA model.

🛠️ What you will create
A fine-tuned language model that generates text based on a specific dataset of quotes, using LoRA.

Dataset
The “Abirate/english_quotes” dataset, specifically a 10% sample of the training split.


In [25]:
# 1. Install necessary libraries (PEFT, datasets).
%pip install peft==0.4.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 658.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 35.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [1]:
# 1.1 Create local folder for model storing :
!mkdir -p cache   # mkdir -p cache creates a "cache" named folder where fine-tuned models can be stored.

In [5]:
from transformers import AutoModelForCausalLM

# Load foundation model
foundation_model = AutoModelForCausalLM.from_pretrained("gpt2")  # ou mistralai/Mistral-7B, etc.

# Automatiquely finding appropriate modules
target_modules = []
for name, module in foundation_model.named_modules():
    if "attn" in name or "dense" in name:
        target_modules.append(name.split('.')[-1])

# Remove duplicates if any
target_modules = list(set(target_modules))

print("Target modules for LoRA:", target_modules)

Target modules for LoRA: ['c_attn', 'c_proj', 'attn_dropout', 'resid_dropout', 'attn']


In [6]:
# 2. Load a pre-trained language model (bigscience/bloomz-560m) and its tokenizer.
%pip install datasets         # This library makes it easy to load public datasets such as "Abirate/english_quotes" (the dataset used here).

In [7]:
# 2.1 - Importer AutoModelForCausalLM, AutoTokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

# 2.2 Pre-trained model name :
model_name = "bigscience/bloomz-560m"

# 2.3. Load tokenizer which corresponds to this model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2.4 Load pre-trained model (foundation model) for purpose of text generation (Causal LM)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

  Task :
    Install necessary libraries (PEFT, datasets0).
    Load a pre-trained language model (bigscience/bloomz-560m) and its tokenizer.
    Load the dataset and preprocess it for the model.
    Configure LoRA using LoraConfig.
    Apply LoRA to the pre-trained model using get_peft_model.
    Set up training arguments using TrainingArguments.
    Initialize and train the model using Trainer.
    Save the fine-tuned LoRA model.
    Load the saved LoRA model for inference using PeftModel.from_pretrained.
    Generate text using the fine-tuned model and the tokenizer.

In [8]:
# 3. Load the dataset and preprocess it for the model.
      # Loading with error handling and fallback option
from datasets import load_dataset
import pandas as pd

def load_quotes_dataset():
    try:
        # Tentative de chargement normal
        dataset = load_dataset("Abirate/english_quotes", split="train")
        return dataset
    except ValueError as e:
        print(f"Erreur lors du chargement: {e}")
        print("Tentative avec une approche alternative...")

        try:
            # Alternative 1: Chargement sans spécifier le split
            dataset = load_dataset("Abirate/english_quotes")
            return dataset["train"] if "train" in dataset else dataset
        except Exception as e2:
            print(f"Deuxième tentative échouée: {e2}")

            try:
                # Alternative 2: Chargement avec cache désactivé
                dataset = load_dataset("Abirate/english_quotes", split="train", cache_dir=None)
                return dataset
            except Exception as e3:
                print(f"Troisième tentative échouée: {e3}")
                return None

# Utilisation
dataset = load_quotes_dataset()

if dataset is not None:
    print(f"Dataset chargé avec succès! Nombre d'éléments: {len(dataset)}")
    print("Exemple:", dataset[0])
else:
    print("Impossible de charger le dataset avec les méthodes automatiques.")


Erreur lors du chargement: Invalid pattern: '**' can only be an entire path component
Tentative avec une approche alternative...
Deuxième tentative échouée: Invalid pattern: '**' can only be an entire path component
Troisième tentative échouée: Invalid pattern: '**' can only be an entire path component
Impossible de charger le dataset avec les méthodes automatiques.


In [9]:
# Load the dataset and preprocess it for the model.

# 3.1 Load 10% of the training dataset
data = load_dataset("Abirate/english_quotes", split="train[:10%]")

# 3.2 Tokenize the text of each quote (field "quote")
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

# 3.3 Verification: display the first 5 tokenized elements
train_sample = data.select(range(5))
display(train_sample)

ValueError: Invalid pattern: '**' can only be an entire path component

In [12]:
# 4. Configure LoRA using LoraConfig.
from peft import LoraConfig

# LoRA configuration :
lora_config = LoraConfig(
    r=1,                          # Rank of the low-rank matrices
    lora_alpha=1,                 # Scaling factor (usually equal to r when r=1)
    target_modules=[              # Target modules to adapt with LoRA
        "query_key_value",        # Query, Key, Value projection
        "dense"                   # Output projection
    ],
    lora_dropout=0.1,             # Dropout for regularization
    bias="none",                  # Do not train bias parameters
    task_type="CAUSAL_LM"         # Task type: causal language modeling
)

print("Configuration LoRA créée avec succès!")
print(f"Rang (r): {lora_config.r}")
print(f"Alpha: {lora_config.lora_alpha}")
print(f"Modules cibles: {lora_config.target_modules}")
print(f"Dropout: {lora_config.lora_dropout}")

Configuration LoRA créée avec succès!
Rang (r): 1
Alpha: 1
Modules cibles: ['query_key_value', 'dense']
Dropout: 0.1


In [13]:
#5. Apply LoRA to the pre-trained model using get_peft_model

from peft import get_peft_model

#Add the adapter layers to the foundation model to be trained
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())

# Apply LoRA to the foundation model
model = get_peft_model(foundation_model, lora_config)

trainable params: 147,456 || all params: 559,362,048 || trainable%: 0.026361459546143537
None


In [14]:
from transformers import TrainingArguments  # ✅ Import nécessaire

# 6. Set up training arguments using TrainingArguments
training_args = TrainingArguments(
    output_dir="./lora-bloomz-quotes",     # Dossier de sortie
    overwrite_output_dir=True,             # Écrase le dossier s’il existe déjà
    report_to="none",                      # Désactive les rapports (W&B, TensorBoard)
    auto_find_batch_size=True,             # Détecte automatiquement la batch size optimale
    learning_rate=3e-2,                    # Taux d’apprentissage (adapté pour LoRA)
    num_train_epochs=3,                    # Nombre d’époques d’entraînement
    use_cpu=True                           # Utilise le CPU (met False si tu veux utiliser un GPU)
)

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType



In [16]:
# 7. Initialize and train the model using Trainer.
        # Fill out the `Trainer` class.

import transformers
from transformers import TrainingArguments, Trainer, AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType
import os

base_model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(base_model_name)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

output_directory = os.path.join("../cache/working", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate= 3e-2, # Higher learning rate than full fine-tuning.
    num_train_epochs=3,
    use_cpu=True
)


In [17]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=4,
    lora_alpha=4,
    target_modules=["c_attn", "c_proj"],  # adapté à GPT2, à changer pour d'autres modèles
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

peft_model = get_peft_model(model, lora_config)
print(peft_model.print_trainable_parameters())

/usr/local/lib/python3.11/dist-packages/peft/tuners/lora.py:299: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 405,504 || all params: 124,845,312 || trainable%: 0.32480514766946156
None


In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    learning_rate=2e-4,
    report_to="none"
)

In [19]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # False pour causal LM (ex : GPT, Mistral)
)

In [25]:
from transformers import Trainer

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,   # doit être un Dataset Hugging Face
    data_collator=data_collator
)

trainer.train()

NameError: name 'data' is not defined

Clara Martinez
  14 h 53
Points à explorer
Importations
Recherchez quelles bibliothèques sont nécessaires pour travailler avec les modèles Hugging Face
Chargement du modèle
Qu'est-ce qu'un modèle causal (AutoModelForCausalLM)?
À quoi sert un tokenizer?
Données
Comment fonctionne la méthode load_dataset?
Que signifie le paramètre split="train[:10%]"?
Prétraitement
Pourquoi utilise-t-on la fonction map sur le dataset?
Comment fonctionne une fonction lambda?
Échantillonnage
Comment sélectionner un petit nombre d'exemples?
Quelle est la différence entre select et d'autres méthodes d'échantillonnage?
Conseils de débogage
Vérifiez la structure de vos données avec print(data)
Explorez les attributs disponibles avec dir(data)
En cas d'erreur, lisez attentivement le message - il contient souvent la solution!
14 h 54
# Mots-clés et fonctions importantes (dans l'ordre chronologique)
1. **import** - Pour accéder aux bibliothèques
   * `datasets`
   * `transformers`
2. **load_dataset** - Fonction pour charger un jeu de données
   * Paramètre `split` - Pour sélectionner une portion
3. **AutoTokenizer.from_pretrained** - Charger un tokenizer
   * Sert à convertir le texte en tokens
4. **AutoModelForCausalLM.from_pretrained** - Charger un modèle de langage
   * Modèle pour la génération de texte
5. **map** - Méthode pour transformer chaque élément d'un dataset
   * `lambda` - Fonction anonyme pour les transformations simples
6. **tokenizer** - Fonction pour tokeniser le texte
   * Transforme le texte en représentation numérique
7. **select** - Méthode pour extraire des échantillons spécifiques
   * `range()` - Fonction pour générer une séquence d'indices
8. **display** - Fonction pour visualiser les données
   * Affiche les résultats de manière formatée

In [ ]:
# 7. Initialize and train the model using Trainer.
from transformers import Trainer

# Initialize Trainer
trainer = Trainer(
    model=model,                         # PEFT model
    args=pythontraining_args,            # Training arguments
    train_dataset=data,                  # Training dataset
    data_collator=lambda data: {'input_ids': torch.stack([x['input_ids'] for x in data]),
                                 'attention_mask': torch.stack([x['attention_mask'] for x in data]),
                                 'labels': torch.stack([x['input_ids'] for x in data])} # Data collator
)

# Start training
trainer.train()

In [ ]:
# 8. Save the fine-tuned LoRA model.
trainer.model.save_pretrained("./lora-bloomz-quotes")

In [ ]:
# 9. Load the saved LoRA model for inference using PeftModel.from_pretrained.
from peft import PeftModel

# Load the PEFT model
loaded_model = PeftModel.from_pretrained(foundation_model, "./lora-bloomz-quotes")

In [ ]:
# 10. Generate text using the fine-tuned model and the tokenizer.
# Helper function to generate text
def generate_text(model, tokenizer, prompt, max_length=50):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    # If you have a GPU, move the input_ids to the GPU
    if torch.cuda.is_available():
        input_ids = input_ids.to("cuda")
        model.to("cuda")

    output = model.generate(input_ids, max_length=max_length, num_return_sequences=1)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Example usage
prompt = "Life is like a box of chocolates"
generated_text = generate_text(loaded_model, tokenizer, prompt)
print(generated_text)